# 案例：任务拆解、依赖排序与冲突重规划

> Last updated: 2026-11-21

## 核心学习目标

本案例以北京3天2晚亲子游请求为对象，学习如何根据任务分解阶段可使用的专业能力把包含预算、景点、体力和天气要求的复杂目标拆解为可执行子任务，再根据各子任务之间以及能力输入、输出定义之间的数据关系分析依赖，最终形成带执行顺序的计划；当运行时出现新冲突时，进一步判断应修订计划还是报告不可行。

## 案例设计

案例按照“理解用户目标—加载能力目录—任务分解—依赖分析—生成候选行程—冲突重规划”的顺序展开。任务分解提示词要求模型保留城市、日期、预算和偏好等槽位；依赖分析提示词要求模型比较子任务输入与技能输出定义；行程生成与修订阶段则把任务级计划转化为读者可以直接检查的逐日安排。

模型创建和调用统一位于代码仓库根目录的 `llm.py`。任务分解、依赖分析、候选行程生成和两类冲突判断都通过真实模型调用完成，不读取预设模型响应。

**学习重点**：
1. 理解专业能力描述如何约束任务分解边界。
2. 掌握把用户 Query 转换为带完整槽位子任务的方法。
3. 理解如何根据能力的技能输入、输出定义识别必要依赖。
4. 掌握把模型返回的依赖映射和 JSON 顺序校验为完整执行计划的方法。
5. 理解如何保留未受影响任务，并区分可重规划冲突与不可解决冲突。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 模型访问 | 环境加载、模型创建、文本提取与 JSON 解析 | 根目录 `llm.py` |
| 任务分解 | 根据能力目录生成带槽位的子任务 | `task_decomposition_prompt_template`、`parse_decomposition_response` |
| 依赖分析 | 根据技能输入、输出定义生成执行顺序 | `dependency_analysis_system_prompt`、`invoke_json` |
| 执行计划 | 校验依赖与顺序并形成版本化初始计划 | `build_initial_plan` |
| 候选行程 | 生成逐日活动、交通、步数与预算估计 | `generate_travel_plan` |
| 冲突重规划 | 根据运行时事件修订任务与具体行程，或报告不可行 | `replan_after_conflict`、`revise_travel_plan` |

## 0. 环境

本案例直接调用所配置的真实模型，不使用fixture或预设响应。运行 Notebook 前，请在本章代码目录执行：

```bash
python -m pip install -r requirements.txt
```

本章复用代码仓库根目录的 `llm.py` 和 `.env`。在代码仓库根目录的 `.env` 或进程环境中配置模型凭据：可用 OpenAI 兼容服务的 `OPENAI_API_KEY`、`OPENAI_BASE_URL`、`OPENAI_MODEL`，也可用通义千问（DashScope）的 `DASHSCOPE_API_KEY`、`DASHSCOPE_CHAT_BASE_URL`、`DASHSCOPE_CHAT_MODEL`；具体使用哪个模型由根目录 `llm.py` 根据已配置的凭据自动选择。

In [1]:
import sys
from pathlib import Path

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

chapter4_candidate = Path.cwd()
if not (chapter4_candidate / "task_decomposition.py").exists():
    chapter_candidates = [
        path for path in Path.cwd().iterdir()
        if path.is_dir()
        and (path / "task_decomposition.py").exists()
        and (path / "conflict_replanning.py").exists()
    ]
    if len(chapter_candidates) != 1:
        raise FileNotFoundError("请从本章代码目录或代码仓库根目录启动Notebook")
    chapter4_candidate = chapter_candidates[0]

book_root_candidate = chapter4_candidate.parent
if not (book_root_candidate / "llm.py").exists():
    raise FileNotFoundError("Could not locate repository-root llm.py")

sys.path.insert(0, str(chapter4_candidate))
sys.path.insert(0, str(book_root_candidate))

from llm import model_configured

print("本章代码目录：已定位")
print("共享模型配置：已加载")
print(f"模型凭据：{'已配置' if model_configured() else '未配置'}")

本章代码目录：已定位
共享模型配置：已加载
模型凭据：已配置


## 1. 定义用户请求

本案例使用教材中的北京亲子游请求，用户目标中同时包含天数、预算、必去景点、体力偏好和雨天调整要求。后续任务分解需要尽量保留这些约束，并且只能使用能力目录中已经声明的能力。

In [2]:
# 可替换为你的测试请求
query = "我想带父母和孩子去北京玩3天2晚，预算8000元以内。希望去故宫、长城和一个适合孩子的科技类场馆。父母不适合长时间步行，每天步行最好不要超过8000步。如果遇到下雨，希望自动调整为室内项目。请给出每天的行程安排、交通建议、预算估计和备选方案。"

## 2. 初始化模型客户端

模型配置和所有模型调用都由代码仓库根目录的共享 `llm.py` 统一封装。本单元直接创建模型客户端（具体模型由根目录 `.env` 决定）；后续任务分解、依赖分析、候选行程生成和冲突重规划都会调用真实模型。缺少模型凭据时，`create_llm()`会给出明确错误。

In [3]:
from llm import create_llm, invoke_chat, invoke_json

llm = create_llm()
print(f"✓ 模型客户端已初始化: {llm.model_name}")

✓ 模型客户端已初始化: qwen3.7-plus


## 3. 定义任务分解能力目录

本案例中的能力目录是提供给任务拆解大模型的**可用能力清单**。它列出当前可使用的专业能力、每项能力适合完成什么任务，以及相关技能的输入与输出定义。

任务拆解器本身不会执行子任务，它依据能力目录判断用户目标能否被现有能力覆盖，再把目标分解为多个可执行子任务。每个条目包含：

- `name` 与 `description`：说明团队中存在哪类专业能力，用于任务分解。
- `skills.inputSchema`：说明完成技能需要哪些信息，用于检查槽位是否完整。
- `skills.outputSchema`：说明技能会产出哪些结果，用于依赖分析。

因此，本章可以把能力目录理解为“任务规划阶段可以使用的能力清单”。这里重点讨论如何借助这份清单完成规划，而不展开智能体实例如何创建、如何路由、如何互相通信以及如何并行执行。

全书案例最终是一套完整的多智能体编排系统：第 6 章会说明这些能力如何由具体子智能体承担并形成不同编排模式。

In [4]:
from typing import Any

CAPABILITY_CATALOG = [
    {
        "name": "weather_forecast",
        "description": "查询指定城市、日期的天气预报。",
        "skills": [
            {
                "name": "get_weather",
                "inputSchema": ["city", "date"],
                "outputSchema": ["weather_summary", "temperature"],
            }
        ],
    },
    {
        "name": "attraction_recommendation",
        "description": "根据城市和偏好推荐旅游景点。",
        "skills": [
            {
                "name": "recommend_attractions",
                "inputSchema": ["city", "preferences"],
                "outputSchema": ["attraction_list"],
            }
        ],
    },
    {
        "name": "hotel_recommendation",
        "description": "根据城市、位置偏好和预算推荐酒店住宿。",
        "skills": [
            {
                "name": "recommend_hotels",
                "inputSchema": ["city", "location_preference", "budget_per_night"],
                "outputSchema": ["hotel_list"],
            }
        ],
    },
    {
        "name": "restaurant_recommendation",
        "description": "根据城市、菜系偏好和人均预算推荐当地特色餐厅。",
        "skills": [
            {
                "name": "recommend_restaurants",
                "inputSchema": ["city", "cuisine_preference", "budget_per_person"],
                "outputSchema": ["restaurant_list"],
            }
        ],
    },
    {
        "name": "flight_search",
        "description": "查询出发地到目的地的航班信息。",
        "skills": [
            {
                "name": "search_flights",
                "inputSchema": ["origin", "destination", "date"],
                "outputSchema": ["flight_list"],
            }
        ],
    },
    {
        "name": "itinerary_planning",
        "description": "根据天数、天气、景点、住宿、餐饮和出行偏好生成行程及备选方案。",
        "skills": [
            {
                "name": "plan_itinerary",
                "inputSchema": [
                    "city_route",
                    "days",
                    "dates",
                    "weather_summary",
                    "attraction_list",
                    "hotel_list",
                    "restaurant_list",
                    "flight_list",
                ],
                "outputSchema": ["itinerary"],
            }
        ],
    },
    {
        "name": "local_transport_planning",
        "description": "根据酒店、景点、行程和行动便利需求规划市内交通并估算每日步行负担。",
        "skills": [
            {
                "name": "plan_local_transport",
                "inputSchema": [
                    "city",
                    "hotel_list",
                    "itinerary",
                    "mobility_preferences",
                ],
                "outputSchema": ["local_transport_plan", "daily_walking_estimates"],
            }
        ],
    },
    {
        "name": "travel_budget_estimation",
        "description": "根据人数、住宿、餐饮、景点和交通方案估算旅行预算并检查预算上限。",
        "skills": [
            {
                "name": "estimate_travel_budget",
                "inputSchema": [
                    "traveler_count",
                    "hotel_list",
                    "restaurant_list",
                    "flight_list",
                    "itinerary",
                    "local_transport_plan",
                    "budget_limit",
                ],
                "outputSchema": ["budget_breakdown", "budget_check"],
            }
        ],
    },
]


def build_capability_catalog_text(capabilities: list[dict]) -> str:
    """Render capability descriptions for task decomposition."""
    team = ""
    for capability in capabilities:
        team += f"{capability['name']}：{capability['description']}\n    "
    return team


def build_capability_parameter_text(capabilities: list[dict]) -> str:
    """Render skill input/output definitions for dependency analysis."""
    text = ""
    for capability in capabilities:
        name = capability.get("name", "")
        text += f"{name}\n"
        for skill in capability.get("skills", []):
            text += (
                f"\t{skill['name']}, "
                f"inputSchema:{skill.get('inputSchema', [])}, "
                f"outputSchema:{skill.get('outputSchema', [])}\n"
            )
    return text


capability_catalog: list[dict[str, Any]] = CAPABILITY_CATALOG
# Natural-language capability view used by the task decomposition prompt.
capability_catalog_text = build_capability_catalog_text(capability_catalog)
# Structured input/output view used by the dependency analysis prompt.
capability_parameter_text = build_capability_parameter_text(capability_catalog)
print("--- 能力目录（任务分解用）---")
print(
    capability_catalog_text[:800]
    + ("..." if len(capability_catalog_text) > 800 else "")
)

--- 能力目录（任务分解用）---
weather_forecast：查询指定城市、日期的天气预报。
    attraction_recommendation：根据城市和偏好推荐旅游景点。
    hotel_recommendation：根据城市、位置偏好和预算推荐酒店住宿。
    restaurant_recommendation：根据城市、菜系偏好和人均预算推荐当地特色餐厅。
    flight_search：查询出发地到目的地的航班信息。
    itinerary_planning：根据天数、天气、景点、住宿、餐饮和出行偏好生成行程及备选方案。
    local_transport_planning：根据酒店、景点、行程和行动便利需求规划市内交通并估算每日步行负担。
    travel_budget_estimation：根据人数、住宿、餐饮、景点和交通方案估算旅行预算并检查预算上限。
    


## 4. 第一步：任务分解提示词

提示词中的 `{}` 依次填入**背景信息**、**能力目录**和**用户输入**。这里将能力目录作为任务拆解的约束边界：分解结果只能使用目录中已经声明的能力，不能凭空增加目录之外的专业能力。后续依赖分析还会使用同一份目录中的技能输入、输出定义。

In [5]:
task_decomposition_prompt_template = """
##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及能力目录中所有可用能力的描述，通过将用户输入分解为能力目录可以覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    {}
##可用能力目录（可用于完成子任务的候选能力）：
    {}
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配能力目录中单一能力的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：逐个判断每个子任务能否由能力目录中的某项能力完成；只有某个子任务确实无法覆盖时，才将该子任务标记为'NULL'，能覆盖的子任务必须保留，不得把全部子任务统一标记为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠能力目录中声明的技能完成，避免擅自扩展为目录之外的能力并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务；
    7.能力目录中的不同能力可以组合完成复杂目标。某一个子任务不可覆盖时，只处理该子任务，不能用'NULL'否定其他可完成子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    {}
"""

background_info = (
    "用户没有提供具体出行日期、出发城市、儿童年龄，也没有给出住宿和餐饮的子预算。"
    "这些缺失信息不得被静默补全为用户事实；需要相关槽位时，应明确写为待用户确认，"
    "或说明只能在信息补充后执行。8000元是总预算上限，不得擅自拆分为固定的住宿、"
    "餐饮或交通预算。能力目录表示规划阶段可使用的能力，不代表已经获得真实天气、"
    "票价、酒店价格、步数或交通数据。"
)
task_decomposition_prompt = task_decomposition_prompt_template.format(
    background_info,
    capability_catalog_text,
    query,
)
print(f"任务分解提示词字符数: {len(task_decomposition_prompt)}")

任务分解提示词字符数: 1878


## 5. 展示：拼好后的「任务分解」完整提示词

下面就是第一步要发给模型的 **user 消息**。

In [6]:
from IPython.display import Markdown, display

display(Markdown(f"**字符数：** {len(task_decomposition_prompt)}"))
display(Markdown("```text\n" + task_decomposition_prompt[:2500] + ("\n...(截断)..." if len(task_decomposition_prompt) > 2500 else "") + "\n```"))

**字符数：** 1878

```text

##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及能力目录中所有可用能力的描述，通过将用户输入分解为能力目录可以覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    用户没有提供具体出行日期、出发城市、儿童年龄，也没有给出住宿和餐饮的子预算。这些缺失信息不得被静默补全为用户事实；需要相关槽位时，应明确写为待用户确认，或说明只能在信息补充后执行。8000元是总预算上限，不得擅自拆分为固定的住宿、餐饮或交通预算。能力目录表示规划阶段可使用的能力，不代表已经获得真实天气、票价、酒店价格、步数或交通数据。
##可用能力目录（可用于完成子任务的候选能力）：
    weather_forecast：查询指定城市、日期的天气预报。
    attraction_recommendation：根据城市和偏好推荐旅游景点。
    hotel_recommendation：根据城市、位置偏好和预算推荐酒店住宿。
    restaurant_recommendation：根据城市、菜系偏好和人均预算推荐当地特色餐厅。
    flight_search：查询出发地到目的地的航班信息。
    itinerary_planning：根据天数、天气、景点、住宿、餐饮和出行偏好生成行程及备选方案。
    local_transport_planning：根据酒店、景点、行程和行动便利需求规划市内交通并估算每日步行负担。
    travel_budget_estimation：根据人数、住宿、餐饮、景点和交通方案估算旅行预算并检查预算上限。
    
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配能力目录中单一能力的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：逐个判断每个子任务能否由能力目录中的某项能力完成；只有某个子任务确实无法覆盖时，才将该子任务标记为'NULL'，能覆盖的子任务必须保留，不得把全部子任务统一标记为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠能力目录中声明的技能完成，避免擅自扩展为目录之外的能力并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务；
    7.能力目录中的不同能力可以组合完成复杂目标。某一个子任务不可覆盖时，只处理该子任务，不能用'NULL'否定其他可完成子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    我想带父母和孩子去北京玩3天2晚，预算8000元以内。希望去故宫、长城和一个适合孩子的科技类场馆。父母不适合长时间步行，每天步行最好不要超过8000步。如果遇到下雨，希望自动调整为室内项目。请给出每天的行程安排、交通建议、预算估计和备选方案。

```

## 6. 调用 LLM 做任务分解

本单元使用能力目录和用户请求构造提示词，并调用真实模型生成任务分解结果。模型调用或输出解析失败时直接暴露错误，便于定位提示词、模型或环境配置问题。

In [7]:
task_decomposition_response = await invoke_chat(
    llm,
    task_decomposition_prompt,
    system_prompt="You are a helpful assistant.",
)
print("✓ 已调用 LLM 完成任务分解")

from IPython.display import Markdown, display

display(Markdown("### 模型输出（任务分解）"))
display(Markdown("```text\n" + task_decomposition_response + "\n```"))

✓ 已调用 LLM 完成任务分解


### 模型输出（任务分解）

```text
# 目标
规划北京3天2晚家庭游行程，包含故宫、长城及儿童科技场馆，满足老人低步行负担及雨天备选需求，总预算控制在8000元以内，并提供交通建议与预算估算。

# 任务拆解
- 查询北京待用户确认日期的天气预报，以评估天气对行程的影响。
- 根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。
- 根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。
- 根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。
- 查询从待用户确认出发地到北京的航班信息。
- 根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。
- 根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。
- 根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。
```

## 7. 解析任务分解结果

从 `# 目标` / `# 任务拆解` 段落提取 **totalGoal** 与 **subSteps** 列表。

In [8]:
import re


def parse_decomposition_response(response: str) -> dict:
    """Parse total goal and subtasks from the model response."""
    totalgoal_key = "# 目标"
    substep_key = "# 任务拆解"

    sub_steps = []
    unsupported_steps = []
    goal_lines = []
    reached_tasks = False

    for line in response.splitlines():
        line = line.strip()
        if not reached_tasks:
            if line.startswith(substep_key):
                reached_tasks = True
                continue
            if line.startswith(totalgoal_key):
                continue
            if line:
                goal_lines.append(line)
        elif line.startswith(("-", "*")):
            task = re.sub(r"^[-*]\s+", "", line).strip()
            task = task.rstrip("#").strip()
            if not task:
                continue
            if task.upper() == "NULL" or re.search(r"[（(]\s*NULL\s*[）)]$", task, re.I):
                unsupported_steps.append(task)
                continue
            sub_steps.append(task)

    if not sub_steps:
        raise ValueError("任务分解结果中没有能力目录可覆盖的子任务")
    return {
        "totalGoal": " ".join(goal_lines),
        "subSteps": sub_steps,
        "unsupportedSteps": unsupported_steps,
    }


decomposition = parse_decomposition_response(task_decomposition_response)
print(f"总体目标: {decomposition['totalGoal']}")
print(f"子任务 ({len(decomposition['subSteps'])}):")
for i, step in enumerate(decomposition['subSteps'], 1):
    print(f"  {i}. {step}")
if decomposition['unsupportedSteps']:
    print("能力目录暂不覆盖的任务：")
    for step in decomposition['unsupportedSteps']:
        print(f"  - {step}")

总体目标: 规划北京3天2晚家庭游行程，包含故宫、长城及儿童科技场馆，满足老人低步行负担及雨天备选需求，总预算控制在8000元以内，并提供交通建议与预算估算。
子任务 (8):
  1. 查询北京待用户确认日期的天气预报，以评估天气对行程的影响。
  2. 根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。
  3. 根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。
  4. 根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。
  5. 查询从待用户确认出发地到北京的航班信息。
  6. 根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。
  7. 根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。
  8. 根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。


## 8. 第二步：依赖分析提示词

为每个子任务分配临时 ID（T1、T2…），把 **subtasks** 与**能力目录中的技能参数**填入 user prompt；system prompt 同时要求模型返回 `depends_on` 与 `execution_order`，使初始计划保留依赖映射和执行顺序。

In [9]:
# 依赖分析提示词保留在 Notebook 中，后续迁移到 Chapter-6 时抽取为独立模块。
dependency_analysis_system_prompt = """
你是一个任务规划系统中的[任务依赖分析器], 负责根据子任务与候选能力的技能输入输出定义, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. capabilities: 候选能力列表, 每个能力提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪项能力的哪个技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 同时输出直接依赖映射与执行顺序，使用如下格式:
{
  "depends_on": {"T1": [], "T2": ["T1"]},
  "execution_order": {"1": "T1", "2": "T2"}
}
- depends_on必须包含全部task_id，只记录直接且必要的依赖。
- execution_order从1开始递增，task_id必须与输入子任务对应。
"""

dependency_analysis_user_prompt_template = """
subtasks:
{subtasks}

capabilities:
{capabilities}

请严格按照输出格式返回, 除此之外不要有其他输出
"""

sub_steps = decomposition["subSteps"]
id_to_task = {f"T{i + 1}": task for i, task in enumerate(sub_steps)}

dependency_analysis_user_prompt = dependency_analysis_user_prompt_template.format(
    subtasks=id_to_task,
    capabilities=capability_parameter_text,
)

print("子任务 ID 映射:")
for tid, desc in id_to_task.items():
    print(f"  {tid}: {desc}")
print(f"\n依赖分析用户提示词字符数: {len(dependency_analysis_user_prompt)}")

子任务 ID 映射:
  T1: 查询北京待用户确认日期的天气预报，以评估天气对行程的影响。
  T2: 根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。
  T3: 根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。
  T4: 根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。
  T5: 查询从待用户确认出发地到北京的航班信息。
  T6: 根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。
  T7: 根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。
  T8: 根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。

依赖分析用户提示词字符数: 1671


## 9. 展示：依赖分析完整提示词

In [10]:
display(Markdown("### System Prompt"))
display(Markdown("```text\n" + dependency_analysis_system_prompt.strip() + "\n```"))
display(Markdown("### User Prompt"))
display(Markdown("```text\n" + dependency_analysis_user_prompt + "\n```"))

### System Prompt

```text
你是一个任务规划系统中的[任务依赖分析器], 负责根据子任务与候选能力的技能输入输出定义, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. capabilities: 候选能力列表, 每个能力提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪项能力的哪个技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 同时输出直接依赖映射与执行顺序，使用如下格式:
{
  "depends_on": {"T1": [], "T2": ["T1"]},
  "execution_order": {"1": "T1", "2": "T2"}
}
- depends_on必须包含全部task_id，只记录直接且必要的依赖。
- execution_order从1开始递增，task_id必须与输入子任务对应。
```

### User Prompt

```text

subtasks:
{'T1': '查询北京待用户确认日期的天气预报，以评估天气对行程的影响。', 'T2': '根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。', 'T3': '根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。', 'T4': '根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。', 'T5': '查询从待用户确认出发地到北京的航班信息。', 'T6': '根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。', 'T7': '根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。', 'T8': '根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。'}

capabilities:
weather_forecast
	get_weather, inputSchema:['city', 'date'], outputSchema:['weather_summary', 'temperature']
attraction_recommendation
	recommend_attractions, inputSchema:['city', 'preferences'], outputSchema:['attraction_list']
hotel_recommendation
	recommend_hotels, inputSchema:['city', 'location_preference', 'budget_per_night'], outputSchema:['hotel_list']
restaurant_recommendation
	recommend_restaurants, inputSchema:['city', 'cuisine_preference', 'budget_per_person'], outputSchema:['restaurant_list']
flight_search
	search_flights, inputSchema:['origin', 'destination', 'date'], outputSchema:['flight_list']
itinerary_planning
	plan_itinerary, inputSchema:['city_route', 'days', 'dates', 'weather_summary', 'attraction_list', 'hotel_list', 'restaurant_list', 'flight_list'], outputSchema:['itinerary']
local_transport_planning
	plan_local_transport, inputSchema:['city', 'hotel_list', 'itinerary', 'mobility_preferences'], outputSchema:['local_transport_plan', 'daily_walking_estimates']
travel_budget_estimation
	estimate_travel_budget, inputSchema:['traveler_count', 'hotel_list', 'restaurant_list', 'flight_list', 'itinerary', 'local_transport_plan', 'budget_limit'], outputSchema:['budget_breakdown', 'budget_check']


请严格按照输出格式返回, 除此之外不要有其他输出

```

## 10. 调用 LLM 做依赖排序

期望返回包含 `depends_on` 和 `execution_order` 的JSON；程序会校验任务引用、依赖关系和拓扑顺序。

In [11]:
import json

def normalize_execution_order(order: object, task_map: dict[str, str]) -> list[str]:
    """Convert model output into a complete execution order."""
    ordered_ids: list[str] = []
    if isinstance(order, dict):
        for value in order.values():
            tid = str(value).strip()
            if tid in task_map and tid not in ordered_ids:
                ordered_ids.append(tid)
    for tid in task_map:
        if tid not in ordered_ids:
            ordered_ids.append(tid)
    return [task_map[tid] for tid in ordered_ids]


if len(sub_steps) < 2:
    dependencies = {task_id: [] for task_id in id_to_task}
    order = {str(index): task_id for index, task_id in enumerate(id_to_task, 1)}
    dependency_payload = {
        "depends_on": dependencies,
        "execution_order": order,
    }
    print("⚠ 子任务少于 2 个，无需调用模型分析依赖")
else:
    dependency_payload = await invoke_json(
        llm,
        dependency_analysis_user_prompt,
        system_prompt=dependency_analysis_system_prompt,
    )
    dependencies = dependency_payload.get("depends_on")
    order = dependency_payload.get("execution_order")
    if not isinstance(dependencies, dict) or not isinstance(order, dict):
        raise ValueError("依赖分析必须返回depends_on和execution_order对象")
    if set(dependencies) != set(id_to_task):
        raise ValueError("depends_on必须恰好覆盖全部子任务")
    print("✓ 已调用 LLM 完成依赖分析与排序")
    print(json.dumps(dependency_payload, ensure_ascii=False, indent=2))

sorted_steps = normalize_execution_order(order, id_to_task)

✓ 已调用 LLM 完成依赖分析与排序
{
  "depends_on": {
    "T1": [],
    "T2": [],
    "T3": [],
    "T4": [],
    "T5": [],
    "T6": [
      "T1",
      "T2",
      "T3",
      "T4",
      "T5"
    ],
    "T7": [
      "T3",
      "T6"
    ],
    "T8": [
      "T2",
      "T3",
      "T4",
      "T5",
      "T6",
      "T7"
    ]
  },
  "execution_order": {
    "1": "T1",
    "2": "T2",
    "3": "T3",
    "4": "T4",
    "5": "T5",
    "6": "T6",
    "7": "T7",
    "8": "T8"
  }
}


## 11. 最终执行计划

In [12]:
result = {
    "total_goal": decomposition["totalGoal"],
    "original_steps": sub_steps,
    "depends_on": dependencies,
    "execution_order": order,
    "sorted_steps": sorted_steps,
}

print("=" * 60)
print("📊 最终执行计划")
print("=" * 60)
print(f"\n总体目标: {result['total_goal']}")
print(f"\n原始子任务顺序:")
for i, step in enumerate(result["original_steps"], 1):
    print(f"  {i}. {step}")
print(f"\n优化后执行顺序:")
for i, step in enumerate(result["sorted_steps"], 1):
    print(f"  {i}. {step}")

📊 最终执行计划

总体目标: 规划北京3天2晚家庭游行程，包含故宫、长城及儿童科技场馆，满足老人低步行负担及雨天备选需求，总预算控制在8000元以内，并提供交通建议与预算估算。

原始子任务顺序:
  1. 查询北京待用户确认日期的天气预报，以评估天气对行程的影响。
  2. 根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。
  3. 根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。
  4. 根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。
  5. 查询从待用户确认出发地到北京的航班信息。
  6. 根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。
  7. 根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。
  8. 根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。

优化后执行顺序:
  1. 查询北京待用户确认日期的天气预报，以评估天气对行程的影响。
  2. 根据北京城市属性及偏好（故宫、长城、适合孩子的科技类场馆）推荐旅游景点。
  3. 根据北京、位置偏好（靠近景点或交通便利）及剩余预算推荐2晚酒店住宿。
  4. 根据北京、菜系偏好（待用户确认）及人均预算推荐当地特色餐厅。
  5. 查询从待用户确认出发地到北京的航班信息。
  6. 根据天数、天气、已选景点、住宿、餐饮、出行偏好（父母不适合长时间步行，每天步行不超过8000步，雨天自动调整为室内项目）生成行程及备选方案。
  7. 根据酒店、景点、行程和行动便利需求（父母不适合长时间步行）规划市内交通并估算每日步行负担。
  8. 根据人数（4人）、住宿、餐饮、景点和交通方案估算旅行预算并检查是否超过8000元上限。


## 12. 构造可维护的初始计划

前面的流程已经得到总体目标、子任务、直接依赖和执行顺序。下面将这些结果整理为带版本号的初始计划，作为候选行程生成与运行时冲突分析的输入。

In [13]:
from conflict_replanning import build_initial_plan, replan_after_conflict

initial_plan = build_initial_plan(
    query,
    decomposition["totalGoal"],
    id_to_task,
    dependencies=dependencies,
    execution_order=order,
)

display(Markdown(
    "### 初始任务级计划\n\n"
    f"- 计划版本：{initial_plan['version']}\n"
    f"- 结构状态：{initial_plan['status']}\n"
    f"- 子任务数：{len(initial_plan['subtasks'])}\n"
    f"- 安全执行顺序：{' → '.join(initial_plan['execution_order'])}"
))

### 初始任务级计划

- 计划版本：1
- 结构状态：ready
- 子任务数：8
- 安全执行顺序：T1 → T2 → T3 → T4 → T5 → T6 → T7 → T8

## 13. 生成读者可检查的候选行程

任务级计划解决了‘需要做哪些工作、按什么顺序做’，但用户最终关心的是三天具体怎样安排。下面将经过校验的任务计划交给行程生成器，由模型生成逐日活动、交通建议、步数区间、预算区间和雨天备选，程序再检查天数、必去项目、步数与预算结构。这里的费用和步数是模型候选估计，待确认信息仍会显式保留。

In [14]:
from travel_plan_generation import (
    build_rain_conflict,
    find_day_by_goal_tag,
    generate_travel_plan,
    revise_travel_plan,
    summarize_travel_changes,
    summarize_travel_plan,
)

travel_plan = await generate_travel_plan(
    llm,
    initial_plan,
    trip_days=3,
    budget_limit_yuan=8000,
    walking_limit_steps=8000,
)

plan_view = summarize_travel_plan(travel_plan)

def table_text(value: object) -> str:
    return str(value).replace("|", "\\|").replace("\n", " ")

rows = [
    "| 天次 | 主题与主要活动 | 交通建议 | 预计步数 | 雨天备选 |",
    "|---|---|---|---|---|",
]
for day in plan_view["days"]:
    rows.append(
        f"| 第{day['day']}天 | {table_text(day['theme'])}："
        f"{table_text(day['activities'])} | "
        f"{table_text(day['transportation'])} | "
        f"{day['walking_steps']} | "
        f"{table_text(day['rainy_day_backup'])} |"
    )
display(Markdown("### 候选三日行程\n\n" + "\n".join(rows)))

pending_labels = {
    "travel_dates": "具体出行日期",
    "departure_city": "出发城市",
    "child_age": "儿童年龄",
    "party_size": "具体人数",
    "traveler_count_confirmation": "具体人数",
    "accommodation_preference": "住宿偏好",
    "hotel_location_preference": "住宿位置偏好",
    "accommodation_budget": "住宿子预算",
    "cuisine_preference": "餐饮偏好",
    "food_budget_and_cuisine_preference": "餐饮子预算与口味偏好",
    "sub_budget_allocation": "分项预算",
    "mobility_needs": "具体行动便利需求",
}
pending_text = "、".join(
    pending_labels.get(field, field)
    for field in plan_view["pending_inputs"]
)
budget = plan_view["budget"]
total = budget["estimated_total_yuan"]
excluded_text = "；".join(budget["excluded_items"])
display(Markdown(
    "### 预算与约束边界\n\n"
    f"- 已计项目的模型估计：{total['min']}--{total['max']}元\n"
    "- 预算状态：待外部核验，不能据此认定完整预算已经通过\n"
    f"- 未计费用：{excluded_text}\n"
    f"- 待确认信息：{pending_text}\n"
    "- 步行偏好：三天估计上界均未超过8000步"
))

### 候选三日行程

| 天次 | 主题与主要活动 | 交通建议 | 预计步数 | 雨天备选 |
|---|---|---|---|---|
| 第1天 | 皇家文化初体验与科技探索：故宫博物院、中国科学技术馆（主展厅） | 地铁1号线转8号线或打车直达，建议打车以节省体力。 | 4000-7000步（模型估计） | 全天在中国科学技术馆内深度参观，或在故宫内延长参观时间并增加休息频次 |
| 第2天 | 长城雄风与休闲调整：慕田峪长城、返回市区，颐和园昆明湖泛舟或长廊漫步 | 建议包车或乘坐旅游专线巴士直达慕田峪，减少换乘折腾。 | 3000-6000步（模型估计） | 取消长城行程，改为参观中国国家博物馆或北京天文馆 |
| 第3天 | 老北京风情与返程准备：天坛公园、前门大街及大栅栏步行街 | 地铁5号线或打车，前门地区交通便利。 | 3000-5000步（模型估计） | 参观北京规划展览馆或在前门附近的老舍茶馆听戏喝茶 |

### 预算与约束边界

- 已计项目的模型估计：3800--5600元
- 预算状态：待外部核验，不能据此认定完整预算已经通过
- 未计费用：往返北京的大交通费用（机票/高铁）；购物及个人额外消费
- 待确认信息：具体出行日期、出发城市、儿童年龄、具体人数、餐饮偏好
- 步行偏好：三天估计上界均未超过8000步

## 14. 第三步：运行时冲突与重新规划

候选行程形成后，运行时事件便可以落到具体日期。代码先从本次模型输出中定位长城所在日，再动态构造该日暴雨且长城临时闭园的事件。模型先修订任务级计划，然后生成具体行程修订；程序会确认长城确实离开闭园日，且未受影响的日期没有被改写。

In [15]:
wall_day = find_day_by_goal_tag(travel_plan, "长城")
conflict_event = build_rain_conflict(wall_day, trip_days=3)

replan_result = await replan_after_conflict(
    llm,
    initial_plan,
    conflict_event,
    capability_parameter_text,
)
if replan_result.get("status") != "replanned":
    raise ValueError("该教学事件应生成可执行的修订计划")

revised_travel_plan = await revise_travel_plan(
    llm,
    travel_plan,
    replan_result,
    conflict_event,
    budget_limit_yuan=8000,
    walking_limit_steps=8000,
)

change_view = summarize_travel_changes(
    travel_plan, revised_travel_plan
)
revised_wall_day = find_day_by_goal_tag(
    revised_travel_plan, "长城"
)
change_rows = [
    "| 天次 | 调整前 | 调整后 | 调整原因 |",
    "|---|---|---|---|",
]
for change in change_view["changed_days"]:
    change_rows.append(
        f"| 第{change['day']}天 | {table_text(change['before'])} | "
        f"{table_text(change['after'])} | {table_text(change['reason'])} |"
    )
display(Markdown("### 具体行程调整\n\n" + "\n".join(change_rows)))
display(Markdown(
    "### 重规划检查\n\n"
    f"- 长城日期：第{wall_day}天 → 第{revised_wall_day}天\n"
    f"- 未变化日期：{', '.join(f'第{day}天' for day in change_view['unchanged_days'])}\n"
    f"- 受影响任务：{'、'.join(replan_result['conflict_analysis']['affected_task_ids'])}\n"
    "- 约束复核：三个必去项目仍被覆盖，步数估计上界仍未超过8000步\n"
    "- 预算复核：已计项目仍为模型估计，完整预算继续等待外部核验"
))

### 具体行程调整

| 天次 | 调整前 | 调整后 | 调整原因 |
|---|---|---|---|
| 第1天 | 故宫博物院、中国科学技术馆（主展厅） | 慕田峪长城、返回市区，颐和园昆明湖泛舟或长廊漫步 | 将长城调整到第1天，避开第2天的暴雨和闭园。 |
| 第2天 | 慕田峪长城、返回市区，颐和园昆明湖泛舟或长廊漫步 | 故宫博物院、中国科学技术馆（主展厅） | 第2天原长城安排遭遇暴雨且景区关闭，改为纯室内项目。 |

### 重规划检查

- 长城日期：第2天 → 第1天
- 未变化日期：第3天
- 受影响任务：T6、T7、T8
- 约束复核：三个必去项目仍被覆盖，步数估计上界仍未超过8000步
- 预算复核：已计项目仍为模型估计，完整预算继续等待外部核验

## 15. 冲突无法解决时的处理

重规划并不保证总能得到可行计划。下面提供一条与原目标直接冲突的运行时事件：用户可出行的三天内所有长城景区均因极端天气和安全管制关闭，同时用户仍坚持实地游览长城且不接受改期、替代、取消或增加天数。规划器应显式报告不可行、列出未解决约束并请求用户决定，而不能虚构仍可执行的路线。

In [16]:
blocking_conflict_event = (
    "运行时收到新信息：在用户可出行的3天内，北京所有对游客开放的长城景区"
    "均因连续极端天气和安全管制关闭，且确认无法恢复；用户仍坚持原3天2晚、"
    "必须实地游览长城，不接受改期、替代景点、取消长城或增加旅行天数。"
    "故宫、科技类场馆和市内服务正常。"
)

infeasible_result = await replan_after_conflict(
    llm,
    initial_plan,
    blocking_conflict_event,
    capability_parameter_text,
)
if infeasible_result.get("status") != "infeasible":
    raise ValueError("该教学事件应被识别为不可解决冲突")

infeasible_view = {
    key: infeasible_result[key]
    for key in (
        "status",
        "reason",
        "unresolved_constraints",
        "requires_user_decision",
        "suggested_actions",
    )
}
print(json.dumps(infeasible_view, ensure_ascii=False, indent=2))

{
  "status": "infeasible",
  "reason": "现有能力无法提供已关闭景区的实地游览服务。由于用户明确拒绝任何替代方案（如替换为其他景点、改期或取消该行程点），在保持原目标不变的前提下，无法生成满足‘必须游览长城’这一约束的可执行计划。",
  "unresolved_constraints": [
    "必须实地游览长城",
    "不接受改期、替代景点、取消长城或增加旅行天数"
  ],
  "requires_user_decision": true,
  "suggested_actions": [
    "请用户同意将‘长城’替换为其他适合老人的室内或低步行负担景点",
    "请用户同意调整出行日期至长城开放的时间段",
    "请用户同意取消长城行程并重新规划剩余时间"
  ]
}


## 16. 小结

- **任务分解**：用户 Query + 能力目录 → 带槽位的子任务列表（`# 目标` / `# 任务拆解`）。
- **依赖排序**：子任务 ID + 能力输入输出 → LLM提出直接依赖与执行顺序 → 程序形成版本化初始计划。
- **候选行程**：任务级计划 → 模型生成逐日活动、交通、步数和预算估计 → 程序校验完整性与约束标记。
- **冲突重规划**：候选行程 + 运行时事件 → 任务级修订与具体行程修订 → 程序确认长城离开闭园日并保留未受影响日期。
- **不可行处理**：当原目标与新冲突无法同时满足时，系统返回未解决约束和可选行动，并请求用户决定。
- **代码边界**：全部候选结果由模型实时生成；程序不使用预设响应，也不在本章执行具体领域智能体。